# CENG463 PA3

In this programming assignment, you will be dealing with encoder-based and decoder-based language models. You will use Python for this task. You can use libraries for your implementations, or implement your own functions. However, you are expected to analyse and reason about your implementation and results.

### IMPORTANT NOTES

* **Do not clear the output of your cells since this notebook will count as your written report and your cell outputs will be used for grading.** If a question in your submitted notebook does not have a printed output, you will get no grade from that question. If you encounter a problem about this, please [email me](mailto:auozturk@ceng.metu.edu.tr) so that we can work out a solution.

* Ideally, you should be able to complete this assignment on Google Colab, without any payment for resources to any service (or you can use your own computers). However, if you believe you need additional resources, please [send an email to Çağrı Hoca](mailto:ctoraman@metu.edu.tr).

In [2]:
import zipfile
import os
# Cannot upload data folder to colab so upload as zip and then unzip.
# Use AI tool to write it very quickly
# Name of your uploaded zip file
zip_file_name = 'data.zip'
extract_path = '.'

# Create the extraction directory if it doesn't exist
os.makedirs(extract_path, exist_ok=True)

# Unzip the file
with zipfile.ZipFile(zip_file_name, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(f'Successfully unzipped {zip_file_name} to {extract_path}')
print('You can now see the contents of the data folder in the files sidebar.')

Successfully unzipped data.zip to .
You can now see the contents of the data folder in the files sidebar.


After running the above cell, you should see your `data` folder and its contents in the Colab file browser. If the `data.zip` file is not in the same directory as your notebook, you might need to adjust `zip_file_name` to include the full path (e.g., `/content/data.zip`).

## Problem: Misinformation Detection

On your last programming assignment, you will experiment with encoder-based and decoder-based models (specifically, BERT and Llama) to identify misinformation on English and Turkish tweet data from 2022. You can read the [paper](https://arxiv.org/pdf/2210.05401v2) for more information about the dataset. However, the dataset that is shared with you with the assignment is slightly simplified, where each row only consists of a `label` and `text` field.

A tweet is considered as "misinformation" if the `label` is `False`. Your models should aim to classify a given text according to the labels `True`, `False`, and `Other`.

**Put the "data" folder in the same directory with your notebook to work on your solutions in case we need to run your notebooks to reproduce your outputs.** The dataset consists of 5 train and test folds. For each model, you will train on 5 folds and report the average performance of all folds.

On the last part of the assignment, you will compare the time and space efficiency of the models you have used. So, don't forget to keep track of the training time.

**Important Note:** For this assignment, using tutorials, online forums, or AI tools to help with debugging and implementation may be needed. However, you are expected to add your resources as disclaimers. **You will be penalized if you do not disclose any help from AI tools, GitHub repositories, or tutorials.**

Tutorials/resources that might come handy:
* [`MiDe22`repository](https://github.com/metunlp/mide22)
* [`transformers` library](https://huggingface.co/docs/transformers/index)
* [finetuning a pretrained model](https://huggingface.co/docs/transformers/training)
* [`pipeline` from HuggingFace for inference](https://huggingface.co/docs/transformers/pipeline_tutorial##text-pipeline)
* [finetuning Llama](https://www.datacamp.com/tutorial/fine-tuning-llama-3-1)
* [quantization for working on low resources](https://huggingface.co/docs/transformers/v5.0.0rc0/quantization/overview)
* Or any other tutorial you find easy to follow. If you find a particularly helpful resource, you can share it in the discussion forum for everyone to use.

## Q1 - Encoder-based language models for classification (25 points)

### Part A: BERT and preprocessing

In this part, you will finetune the `google-bert/bert-base-uncased` model for the misinformation detection task for English tweets ("EN" folder in the dataset). However, you will train two versions: one with preprocessed texts, one with the raw texts.

For the preprocessing steps, lowercase the text, and use `nltk` to lemmatize and remove stopwords. You can also split each tweet into sentences and tokens, then combine the token list into a single space seperated string to represent each tweet as a single text again. Be careful not to combine different tweets. You can add additional preprocessing steps as long as you keep the integrity of the label-text mappings for each tweet.

You can also use the `preprocess_review` function from PA2 and then combine the returned list into a single space separated string.

Report the performance of both classifiers as the average of 5 train/test folds in "EN" dataset and accuracy, precision, recall, and F1-score with respect to the `False` class. Discuss which model performed better and why.

### Part B: BERT, Mutlilingual BERT and crosslingual performance

In this part, you will finetune the `google-bert/bert-base-uncased` and `google-bert/bert-base-multilingual-uncased` models and compare their performances. However, for the train/test folds, you will take the train fold from "EN" folder and the test fold with the corresponding number from the "TR" folder. This means that you will train the models on English but test them on Turkish.

Report the performance of both classifiers as the average of 5 train/test folds on accuracy, precision, recall, and F1-score with respect to the `False` class. Discuss which model performed better and why you think that is the case.



In [3]:
# IMPLEMENT Q1 PART A HERE.
# YOU CAN ADD CELLS BELOW.

In [ ]:
### PART A

In [4]:
# Read the dataset from tsv file
# need to read each test and train tsv for EN and TR folders

import os
import pandas as pd

root = "data"

datasets = {
    "EN": {"train": {}, "test": {}},
    "TR": {"train": {}, "test": {}}
}

language_dict = {
    "EN": "english",
    "TR": "turkish"
}

for lang in language_dict.keys(): # ["EN", "TR"]
    lang_path = os.path.join(root, lang)


    #indexes for datasets
    train_idx = 0
    test_idx = 0

    for file in sorted(os.listdir(lang_path)):   # sorted = stable order
        if file.endswith(".tsv"):
            df = pd.read_csv(os.path.join(lang_path, file), sep="\t")

            # assign numeric order
            if "train" in file.lower():
                datasets[lang]["train"][train_idx] = df
                train_idx += 1
            elif "test" in file.lower():
                datasets[lang]["test"][test_idx] = df
                test_idx += 1

# example access
#print(datasets["EN"]["train"][1].head())
print(datasets["TR"]["train"][4].head())



   label                                               text
0  False  Izmir mv @avhamzadag neden izmirde multecilere...
1  Other  Açız, mutfak masrafları, Kira, okul giderleri,...
2  False  @yyunikonn İzmire yığılma olur diye yorumladim...
3   True  ❌ MEB'in İzmir’de mültecilere özel dört okul y...
4  Other  @wolfintoaction Çok gerekliyse ülkeye göçmen d...


In [5]:
## Preprocessing Steps
import re

# taken from PA1 and apoted for our needs
def remove_usernames(text):
    # valid twitter usernames = @ + 2–15 chars alnum or underscore
    return re.sub(r'@([A-Za-z0-9_]{2,15})', '', text)

def remove_hashtags(text):
    return re.sub(r'#\w+', '', text)

emoji_pattern = re.compile(
    "["
    "\U0001F600-\U0001F64F"  # Emoticons
    "\U0001F300-\U0001F5FF"  # Symbols & pictographs
    "\U0001F680-\U0001F6FF"  # Transport & map symbols
    "\U0001F1E0-\U0001F1FF"  # Flags
    "\U00002700-\U000027BF"  # Dingbats
    "\U0001F900-\U0001F9FF"  # Supplemental Symbols & Pictographs
    "]+",
    flags=re.UNICODE
)

def remove_emojis(text):
    return emoji_pattern.sub('', text)


# this regex taken from AI
def remove_numbers(text):
    # remove standalone numeric-like tokens
    return re.sub(r'\b[\d.,:]+\b', '', text)

def replace_links(text): #replace links with [URL]
    return re.sub(r'http\S+|www\S+|https\S+', '[URL]', text, flags=re.MULTILINE)

# Taken from PA2 and modified for our needs
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('punkt_tab')

def preprocess_review(text, lang='english'):
    # remove unwanted patterns
    text = replace_links(text) # i add this since in datasets there are too many links
    text = remove_usernames(text)
    text = remove_hashtags(text)
    text = remove_emojis(text)
    text = remove_numbers(text)

    # normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    stop_words = set(stopwords.words(lang))
    lemmatizer = WordNetLemmatizer()

    sentences = sent_tokenize(text, language=lang) # it takes language parameter, we did not use it before since we only had english reviews before

    result_tokens = []

    for sentence in sentences:
        tokenized_sentence = word_tokenize(sentence)
        lowercased_sentence = [token.lower() for token in tokenized_sentence]
        stopwords_removed_sentence = [token for token in lowercased_sentence if token not in stop_words]
        lemmatized_sentence = [lemmatizer.lemmatize(token) for token in stopwords_removed_sentence]

        result_tokens.extend(lemmatized_sentence)

    return " ".join(result_tokens)

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [6]:
# PREPROCESSING FUNCTIONS GIVEN FOR ME in PA2

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('punkt_tab')

def preprocess_review_alt(review, lang='english'):
    review = remove_usernames(review)
    review = remove_hashtags(review)
    stop_words = set(stopwords.words(lang))
    lemmatizer = WordNetLemmatizer()
    sentences = sent_tokenize(review)

    lemmatized_review = []

    for sentence in sentences:
        tokenized_sentence = word_tokenize(sentence)
        lowercased_sentence = [token.lower() for token in tokenized_sentence]
        stopwords_removed_sentence = [token for token in lowercased_sentence if token not in stop_words]
        lemmatized_sentence = [lemmatizer.lemmatize(token) for token in stopwords_removed_sentence]

        lemmatized_review = lemmatized_review + lemmatized_sentence

    # single string output
    return ' '.join(lemmatized_review)

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [7]:
# test the preprocessing function
sample_text = "@dpa_live @dpa @onetz_de @_FriedrichMerz @n_roettgen @HBraun @EschleThomas @CDU @CSU @Markus_Soeder @derspiegel @welt @tagesschau @ZDF @SZ @Tagesspiegel @BR24 @maybritillner @AnneWillTalk @heutejournal @cnnbrk @faznet @washingtonpost @nytimes @OlafScholz @ABaerbock @c_lindner @NancyFaeser @MarcoBuschmann @ScottMorrisonMP @karenandrewsmp @ashbarty @AustralianOpen @DjokerNole @BILD @EP_President @DavidSassoli @AlexHawkeMP @PeterDutton_MP @usopen @Olympics @FIFAcom @FIFAWorldCup @Pasferre @francefootball @Cristiano @TomBrady @mosseri @instagram @POTUS #UKRAINE PRES.REFUSED #PEACE INITIATED BY #SCHOLZ FEW DAYS BEFORE #RUSSIA ATTACKED=U RESPONSIBLE f.MANY DEATH+WAR TOO+#EINSTEIN:WORLD BACK #STONEAGE IF WW3+#POLAND SILLY WANT USA #NUCLEAR+GER MOBBERS MULTIPLE MORE SADISTIC TO ME=NO #WAR END YOUR OWN #SINS https://t.co/HYSfo3Nsfg."
preprocessed_text = preprocess_review(sample_text, lang='english')
pre_text_alt = preprocess_review_alt(sample_text, lang='english')
print("Original Text:", sample_text)
print("Preprocessed Text:", preprocessed_text)
print("Preprocessed Text:", pre_text_alt)

Original Text: @dpa_live @dpa @onetz_de @_FriedrichMerz @n_roettgen @HBraun @EschleThomas @CDU @CSU @Markus_Soeder @derspiegel @welt @tagesschau @ZDF @SZ @Tagesspiegel @BR24 @maybritillner @AnneWillTalk @heutejournal @cnnbrk @faznet @washingtonpost @nytimes @OlafScholz @ABaerbock @c_lindner @NancyFaeser @MarcoBuschmann @ScottMorrisonMP @karenandrewsmp @ashbarty @AustralianOpen @DjokerNole @BILD @EP_President @DavidSassoli @AlexHawkeMP @PeterDutton_MP @usopen @Olympics @FIFAcom @FIFAWorldCup @Pasferre @francefootball @Cristiano @TomBrady @mosseri @instagram @POTUS #UKRAINE PRES.REFUSED #PEACE INITIATED BY #SCHOLZ FEW DAYS BEFORE #RUSSIA ATTACKED=U RESPONSIBLE f.MANY DEATH+WAR TOO+#EINSTEIN:WORLD BACK #STONEAGE IF WW3+#POLAND SILLY WANT USA #NUCLEAR+GER MOBBERS MULTIPLE MORE SADISTIC TO ME=NO #WAR END YOUR OWN #SINS https://t.co/HYSfo3Nsfg.
Preprocessed Text: presrefused initiated day attacked=u responsible fmany death+war too+ : world back ww3+ silly want usa +ger mobbers multiple sadis

In [8]:
#turkish test
sample_text_tr = "Hangi akla hizmetse" + """
Karşıyaka Bostanlı da
Kiraların 3.000 TL olduğu
1 tane Suriyeli nin yaşamadığı
Bostanlının göbeğinde
Yıllardır okul yapılmayan arsaya
Suriyeliler için
Mülteci Okulu yapıyorlar

Okulumu yıktılar https://t.co/cGflKdPzLa #gazetesozcu @gazetesozcu"""
preprocessed_text = preprocess_review(sample_text_tr, lang='turkish')
pre_text_alt = preprocess_review_alt(sample_text_tr, lang='turkish')
print("Preprocessed Text:\n", preprocessed_text)
print("Preprocessed Text:\n", pre_text_alt)

Preprocessed Text:
 hangi akla hizmetse karşıyaka bostanlı kiraların tl olduğu tane suriyeli nin yaşamadığı bostanlının göbeğinde yıllardır okul yapılmayan arsaya suriyeliler mülteci okulu yapıyorlar okulumu yıktılar [ url ]
Preprocessed Text:
 hangi akla hizmetse karşıyaka bostanlı kiraların 3.000 tl olduğu 1 tane suriyeli nin yaşamadığı bostanlının göbeğinde yıllardır okul yapılmayan arsaya suriyeliler mülteci okulu yapıyorlar okulumu yıktılar http : //t.co/cgflkdpzla


In [9]:
# apply preprocessing to all datasets with new column prep_text
for lang in datasets.keys(): # ["EN", "TR"]
    lang_name = language_dict[lang]
    for split in datasets[lang].keys(): # ["train", "test"]
        for idx in datasets[lang][split].keys():
            df = datasets[lang][split][idx]
            df['prep_text'] = df['text'].apply(lambda x: preprocess_review(x, lang=lang_name))
            df['prep_text_alt'] = df['text'].apply(lambda x: preprocess_review_alt(x, lang=lang_name))
            datasets[lang][split][idx] = df


In [10]:
# check db
print(datasets["EN"]["train"][0]["text"])
print("*"*10)
print(datasets["TR"]["train"][0].head())

0       @benjaminhaddad Now maybe you can rethink the ...
1       Poland, Hungary and now Ukraine via Zelensky a...
2       @therrienv @stupidmaggats @417craig @Yeshua17F...
3       @dpa_live @dpa @onetz_de @_FriedrichMerz @n_ro...
4       @Podolyak_M Americans stand with the people of...
                              ...                        
4206    Wow! I believed it to be true. So much politic...
4207    Today's nominations for Greatest Briton...\n\n...
4208    #RepublicExclusive | 'Endured hardest days of ...
4209    Former Miss Ukraine Anastasiia Lenna has clari...
4210    MISS UKRAINE SUITS UP TO DEFEND HOMELAND\nShe ...
Name: text, Length: 4211, dtype: object
**********
   label                                               text  \
0   True  MEB'den İzmir'deki okul için açıklama: Ağaçlar...   
1  False  @KORDELYAz okul yapılıyor ama bir fark var. mü...   
2   True  'Mültecilere okul' yalanıyla kışkırtma peşinde...   
3   True  **İstiklal madalyası sahibi Milli Eğ. Bak. Mus...

Model class

In [25]:
import random
import numpy as np

from datasets import DatasetDict, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

from sklearn.metrics import precision_recall_fscore_support, classification_report
from sklearn.metrics import accuracy_score



In [26]:
CONFIG = {
    "train_lang": "EN",
    "test_lang": "EN",
    "split_idx": 0,

    "model_name": "bert-base-uncased",
    "num_labels": 3,
    "max_length": 128,

    "batch_size": 16,
    "epochs": 10,
    "lr": 5e-5,
    "pretrained": 1,
}

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    p, r, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted"
    )

    return {
        "precision": p,
        "recall": r,
        "f1": f1,
    }

In [ ]:
# Add this cell before build_dataset_dict function
label_map = {
    "True": 0,
    "False": 1,
    "Other": 2
}

def build_dataset_dict(datasets, cfg):
    train_df = datasets[cfg["train_lang"]]["train"][cfg["split_idx"]].copy()
    test_df  = datasets[cfg["test_lang"]]["test"][cfg["split_idx"]].copy()

    # Map string labels to integers
    train_df["label"] = train_df["label"].map(label_map)
    test_df["label"] = test_df["label"].map(label_map)

    # Drop any rows with NaN labels (in case there are unexpected label values)
    train_df = train_df.dropna(subset=["label"])
    test_df = test_df.dropna(subset=["label"])

    # Convert to int
    train_df["label"] = train_df["label"].astype(int)
    test_df["label"] = test_df["label"].astype(int)

    return DatasetDict({
        "train": Dataset.from_pandas(train_df, preserve_index=False),
        "test":  Dataset.from_pandas(test_df, preserve_index=False),
    })

In [ ]:
def tokenize_dataset(dataset_dict, tokenizer, cfg):
    text_col = "prep_text" if cfg["pretrained"] == 1  else "prep_text_alt" if cfg["pretrained"] == 2 else "text"

    def tokenize_fn(batch):
        return tokenizer(
            batch[text_col],
            truncation=True,
            max_length=cfg["max_length"],
        )

    return dataset_dict.map(tokenize_fn, batched=True)


In [ ]:
def build_trainer(cfg, model, tokenizer, dataset_dict):
    args = TrainingArguments(
        output_dir="./results",
        learning_rate=cfg["lr"],
        per_device_train_batch_size=cfg["batch_size"],
        per_device_eval_batch_size=cfg["batch_size"],
        num_train_epochs=cfg["epochs"],
        report_to=["none"],
        logging_strategy="steps",
        save_strategy="no",
        seed=random.randint(1, 1000),
        disable_tqdm=False
    )

    return Trainer(
        model=model,
        args=args,
        train_dataset=dataset_dict["train"],
        eval_dataset=dataset_dict["test"],
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics,
    )


In [ ]:
def evaluate_false_class(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[1],  # Specify the "False" class
        average=None,  # Return metrics for each label separately
        zero_division=0
    )

    # Since labels=[1], precision, recall, f1 are arrays with one element
    return {
        "accuracy": acc,
        "precision_false": precision[0],
        "recall_false": recall[0],
        "f1_false": f1[0],
    }

In [ ]:
def run_5fold_experiment(datasets, cfg):
    metrics_all = []

    for fold in range(5):
        cfg["split_idx"] = fold  # Set the current fold index
        dataset_dict = build_dataset_dict(datasets, cfg)

        tokenizer = AutoTokenizer.from_pretrained(cfg["model_name"])
        dataset_dict = tokenize_dataset(
            dataset_dict,
            tokenizer,
            cfg  # Pass the full config dict
        )

        model = AutoModelForSequenceClassification.from_pretrained(
            cfg["model_name"],
            num_labels=cfg["num_labels"]
        )

        trainer = build_trainer(cfg, model, tokenizer, dataset_dict)
        trainer.train()

        preds = trainer.predict(dataset_dict["test"])
        y_pred = np.argmax(preds.predictions, axis=1)
        y_true = dataset_dict["test"]["label"]

        fold_metrics = evaluate_false_class(y_true, y_pred)
        metrics_all.append(fold_metrics)

    return metrics_all

In [ ]:
def average_metrics(metrics_list):
    return {
        k: np.mean([m[k] for m in metrics_list])
        for k in metrics_list[0]
    }

In [ ]:
CONFIG["train_lang"] = "EN"
CONFIG["test_lang"] = "EN"
CONFIG["pretrained"] = 1

raw_metrics = run_5fold_experiment(
    datasets,
    cfg=CONFIG
)

raw_avg = average_metrics(raw_metrics)
raw_avg



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Map:   0%|          | 0/4242 [00:00<?, ? examples/s]

Map:   0%|          | 0/1042 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-2767618773.py:15: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  return Trainer(


Step,Training Loss
500,0.591300
1000,0.194100
1500,0.065600
2000,0.018800
2500,0.008700


{'accuracy': np.float64(0.8096725864848784),
 'precision_false': np.float64(0.77111545783116),
 'recall_false': np.float64(0.777976163681617),
 'f1_false': np.float64(0.7742638732179181)}

In [ ]:
CONFIG["pretrained"] = 2 # use from pa2 preprocessed
CONFIG['split_idx'] = 0 #reset the index

raw_metrics = run_5fold_experiment(
    datasets,
    cfg=CONFIG
)

raw_avg = average_metrics(raw_metrics)
raw_avg

Map:   0%|          | 0/4211 [00:00<?, ? examples/s]

Map:   0%|          | 0/1073 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-2767618773.py:15: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  return Trainer(


Step,Training Loss
500,0.596600
1000,0.174200
1500,0.053300
2000,0.029300
2500,0.007000


Map:   0%|          | 0/4220 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/4242 [00:00<?, ? examples/s]

Map:   0%|          | 0/1042 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-2767618773.py:15: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  return Trainer(


Step,Training Loss
500,0.627500
1000,0.218700
1500,0.072600
2000,0.022500
2500,0.005800


{'accuracy': np.float64(0.8072942527719068),
 'precision_false': np.float64(0.7687953523567138),
 'recall_false': np.float64(0.7760489502407119),
 'f1_false': np.float64(0.7719301051102241)}

In [ ]:
CONFIG["pretrained"] = 0 # use raw
CONFIG['split_idx'] = 0 #reset the index

raw_metrics = run_5fold_experiment(
    datasets,
    cfg=CONFIG
)

raw_avg = average_metrics(raw_metrics)
raw_avg

Map:   0%|          | 0/4211 [00:00<?, ? examples/s]

Map:   0%|          | 0/1073 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-2767618773.py:15: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  return Trainer(


Step,Training Loss
500,0.551300
1000,0.143200
1500,0.030700
2000,0.010800
2500,0.002500


Map:   0%|          | 0/4220 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/4242 [00:00<?, ? examples/s]

Map:   0%|          | 0/1042 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-2767618773.py:15: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  return Trainer(


Step,Training Loss
500,0.601800
1000,0.175700
1500,0.054600
2000,0.012500
2500,0.004600


{'accuracy': np.float64(0.8230723603143117),
 'precision_false': np.float64(0.8092369930281794),
 'recall_false': np.float64(0.7806932028879681),
 'f1_false': np.float64(0.7937964902705207)}

**DISCUSS Q1 PART A HERE.**

Discuss PARTA TO DO

In [ ]:
# IMPLEMENT Q1 PART B HERE.
# YOU CAN ADD CELLS BELOW.

In [ ]:
CONFIG["test_lang"] = "TR"
CONFIG["pretrained"] = 0 # use raw
CONFIG['split_idx'] = 0 #reset the index

raw_metrics_TR_bilang = run_5fold_experiment(
    datasets,
    cfg=CONFIG
)

raw_avg_TR_bilang = average_metrics(raw_metrics_TR_bilang)
raw_avg_TR_bilang

Map:   0%|          | 0/4211 [00:00<?, ? examples/s]

Map:   0%|          | 0/1027 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-2767618773.py:15: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  return Trainer(


Step,Training Loss
500,0.543100
1000,0.142300
1500,0.036500
2000,0.005000
2500,0.005100


Map:   0%|          | 0/4220 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/4242 [00:00<?, ? examples/s]

Map:   0%|          | 0/996 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-2767618773.py:15: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  return Trainer(


Step,Training Loss
500,0.553000
1000,0.150100
1500,0.045900
2000,0.016500
2500,0.003200


{'accuracy': np.float64(0.4822720962730691),
 'precision_false': np.float64(0.37847486678026676),
 'recall_false': np.float64(0.37158485114315365),
 'f1_false': np.float64(0.3220291628385561)}

In [ ]:
CONFIG["test_lang"] = "TR"
CONFIG["pretrained"] = 0 # use raw
CONFIG['split_idx'] = 0 #reset the index
CONFIG["model_name"] = "bert-base-multilingual-uncased"

raw_metrics_TR_multilang = run_5fold_experiment(
    datasets,
    cfg=CONFIG
)

raw_avg_TR_multilang = average_metrics(raw_metrics_TR_multilang)
raw_avg_TR_multilang

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/872k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.72M [00:00<?, ?B/s]

Map:   0%|          | 0/4211 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/4242 [00:00<?, ? examples/s]

Map:   0%|          | 0/996 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-2767618773.py:15: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  return Trainer(


Step,Training Loss
500,0.733900
1000,0.327200
1500,0.144600
2000,0.047300
2500,0.015300


{'accuracy': np.float64(0.5140411251129701),
 'precision_false': np.float64(0.331616658736415),
 'recall_false': np.float64(0.22234618459321515),
 'f1_false': np.float64(0.23696676995036708)}

**DISCUSS Q1 PART B HERE.**

## Q2 - Decoder-based language models for classification (55 points)

### Part A: Zero-shot and Few-shot inference with Llama for text classification

In this part, you will use the `meta-llama/Llama-3.1-8B-Instruct` model and the "EN" English tweets dataset for designing a classifier based on inference. You will design two prompts:

* **Prompt 1:** A zero-shot prompt with only the task description, no examples. Explain the task and ask for a classification of the tweets in test folds.
* **Prompt 2:** A one-shot or few-shot prompt, with one or a couple examples. This can be tricky as longer tweets may cause problems. Explain the task using examples from the training folds, and ask for a classification of the tweets in test folds.

Report the performance of both classification pipelines as the average of 5 test folds on accuracy, precision, recall, and F1-score with respect to the `False` class. Explain your prompt design process in detail and discuss the performance of the models with respect to your prompts.

### Part B: Finetuning Llama

In this part, you will again use `meta-llama/Llama-3.1-8B-Instruct` model and the "EN" English tweets dataset but this time you will try to finetune the model with the training folds. However, this may become tricky and resource demanding, considering you are using free tools (such as Google Colab) or your own computers. Check out the tutorials to see how you can better manage your resources.

Report the performance of the finetuned classifier as the average of 5 test folds on accuracy, precision, recall, and F1-score with respect to the `False` class, with the better performing prompt you have designed in Part A. Discuss the effect of finetuning on the model performance. Do you think it improved the performance? What else can be integrated into the process to improve the performance of the model, specifically for the misinformation detection task?


### Part C: Domain transfer to sentiment analysis

In this part, you will try to classify game reviews with the model you finetuned in Part B to see the effect of finetuning on the generalized performance of the model.

You should design a simple zero-shot prompt that asks the model whether a user recommends the game given their review text. With the same prompt, you will use both the base `meta-llama/Llama-3.1-8B-Instruct` model and your finetuned version to classify the reviews.

Report the performance of both classifiers on the whole dataset with respect to accuracy, precision, recall, and F1-score. Discuss and explain your findings.

**Note: Dataset for Part C**

* You will use the `game_reviews.csv` file shared with you. It is taken from [this Kaggle dataset](https://www.kaggle.com/datasets/piyushagni5/sentiment-analysis-for-steam-reviews). Review text attribute is `user_review`. User recommendation is the attribute `user_suggestion` where `1` means user recommended the game, `0` means user did not recommend the game.

In [11]:
# IMPLEMENT Q2 PART A HERE.
# YOU CAN ADD CELLS BELOW.
import torch
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

In [12]:
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map="auto"
)

model.eval()


tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
  

In [13]:
def create_zero_shot_prompt(tweet):
    return f"""
Classify the following tweet as True, False, Other based on whether it contains misinformation.
- True: The tweet is factual and not misleading.
- False: The tweet contains misinformation (false or deceptive information).
- Other: The tweet is neither clearly true nor false

Tweet: "{tweet}"

Classification:
""".strip()


In [14]:
def create_few_shot_prompt(tweet_text, examples):
    prompt = """
Classify the following tweet as "True", "False", or "Other" based on whether it contains misinformation.
- "True": The tweet is factual and not misleading.
- "False": The tweet contains misinformation (false or deceptive information).
- "Other": The tweet is neither clearly true nor false (e.g., opinion or irrelevant).

Examples:
"""
    for ex_text, ex_label in examples:
        prompt += f'- Tweet: "{ex_text}" -> Classification: {ex_label}\n'
    prompt += f"""
Now classify this tweet:
Tweet: "{tweet_text}"

Classification:
"""
    return prompt

In [15]:
# test the prompts
print(create_zero_shot_prompt("this is misinformation"))
print("*" * 30)

examples = [
    ("this is not misinformation", "True")
]
print(create_few_shot_prompt("this is misinformation",examples))

Classify the following tweet as True, False, Other based on whether it contains misinformation.
- True: The tweet is factual and not misleading.
- False: The tweet contains misinformation (false or deceptive information).
- Other: The tweet is neither clearly true nor false

Tweet: "this is misinformation"

Classification:
******************************

Classify the following tweet as "True", "False", or "Other" based on whether it contains misinformation.
- "True": The tweet is factual and not misleading.
- "False": The tweet contains misinformation (false or deceptive information).
- "Other": The tweet is neither clearly true nor false (e.g., opinion or irrelevant).

Examples:
- Tweet: "this is not misinformation" -> Classification: True

Now classify this tweet:
Tweet: "this is misinformation"

Classification:



In [16]:
def parse_classification(response):
    # Extract the label from the generated text (case-insensitive)
    text = response.lower()
    if "false" in text:
        return "False"
    elif "true" in text:
        return "True"
    else:
        return "Other"

In [17]:
def sample_few_shot_examples(train_df, n_examples_per_class):
    """
    Samples up to n_examples_per_class examples for each class:
    False, True, Other.
    Help with AI to get examples randomly
    """
    class_order = ["False", "Other", "True"]
    samples_per_class = {}

    for label in class_order:
        class_df = train_df[train_df["label"] == label]

        if len(class_df) == 0:
            samples_per_class[label] = []
            continue

        sampled = class_df.sample(
            n=min(n_examples_per_class, len(class_df)),
            random_state=42
        )

        samples_per_class[label] = list(
            zip(sampled["text"], sampled["label"])
        )

    examples = []
    for i in range(n_examples_per_class):
        for label in class_order:
            if i < len(samples_per_class[label]):
                examples.append(samples_per_class[label][i])


    return examples

In [18]:
train_df = datasets["EN"]["train"][0]
examples = sample_few_shot_examples(train_df, 1)
print(create_few_shot_prompt("this is misinformation",examples))


Classify the following tweet as "True", "False", or "Other" based on whether it contains misinformation.
- "True": The tweet is factual and not misleading.
- "False": The tweet contains misinformation (false or deceptive information).
- "Other": The tweet is neither clearly true nor false (e.g., opinion or irrelevant).

Examples:
- Tweet: "@Porkchop_EXP I could recreate those exact pictures this afternoon with the help of my friends. Staged war propaganda photos are as old as photography. They might be real or they might not, accepting them at face value is not wise however." -> Classification: False
- Tweet: "And can automatically track you, wherever you are.

New microchip implant stores your COVID vaccine passport https://t.co/KqEG2SUNhm" -> Classification: Other
- Tweet: "@FO_Coalition I’m amazed that you have allowed the Russian embassy to publish this statement online in Canada without a red disclaimer sticker warning of disinformation, and fact check inserts! This is shameful! 

In [28]:
label_map = {
    "True": 0,
    "False": 1,   # ← positive class
    "Other": 2
}

In [47]:
from tqdm import tqdm


def classify_fold_with_prompts(fold_idx=0, shot_number=0):
    test_df = datasets["EN"]["test"][fold_idx]
    train_df = datasets["EN"]["train"][fold_idx]

    predictions = []
    answer_labels = test_df["label"].tolist()

    # ✅ Sample few-shot examples ONCE per fold
    if shot_number > 0:
        few_shot_examples = sample_few_shot_examples(
            train_df,
            n_examples_per_class=shot_number
        )
    else:
        few_shot_examples = None

    for _, row in tqdm(
        test_df.iterrows(),
        total=len(test_df),
        desc=f"Fold {fold_idx} | {shot_number}-shot"
    ): # Thanks to AI, I added process bar for visualization
        tweet = row["text"]

        if shot_number == 0:
            prompt = create_zero_shot_prompt(tweet)
        else:
            prompt = create_few_shot_prompt(tweet, few_shot_examples)

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        input_length = inputs['input_ids'].shape[1]

        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False, # set this and remove temprature etc to get deterministic results
            pad_token_id=tokenizer.eos_token_id #prints something so to get rid of this i added this line thanks to AI
        )

        decoded = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
        #print(f"Full decoded response: {decoded}")  # Add this to inspect
        prediction = parse_classification(decoded)

        predictions.append(prediction)

    # Convert to numeric
    pred_numeric = [label_map[p] for p in predictions]
    true_numeric = [label_map[t] for t in answer_labels]

    y_true_bin = [1 if t == label_map["False"] else 0 for t in true_numeric]
    y_pred_bin = [1 if p == label_map["False"] else 0 for p in pred_numeric]

    # Metrics (False = positive class)
    acc = accuracy_score(y_true_bin, y_pred_bin)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true_bin,
        y_pred_bin,
        average="binary",
        zero_division=0
    )

    return {
        "accuracy": acc,
        "precision_false": precision,
        "recall_false": recall,
        "f1_false": f1,
    }

In [48]:
zero_shot_metrics = []
for fold in range(5):
    metrics = classify_fold_with_prompts(fold, shot_number=0)
    zero_shot_metrics.append(metrics)
    print(f"Zero-shot Fold {fold}: {metrics}")

zero_shot_avg = {k: np.mean([m[k] for m in zero_shot_metrics]) for k in zero_shot_metrics[0]}
print("Zero-shot Average Metrics:", zero_shot_avg)
print("-" * 100)

Fold 0 | 0-shot: 100%|██████████| 1073/1073 [03:16<00:00,  5.47it/s]


Zero-shot Fold 0: {'accuracy': 0.7604846225535881, 'precision_false': 0.6952789699570815, 'recall_false': 0.46551724137931033, 'f1_false': 0.5576592082616179}


Fold 1 | 0-shot: 100%|██████████| 1064/1064 [03:18<00:00,  5.37it/s]


Zero-shot Fold 1: {'accuracy': 0.7349624060150376, 'precision_false': 0.6443514644351465, 'recall_false': 0.43874643874643876, 'f1_false': 0.5220338983050847}


Fold 2 | 0-shot: 100%|██████████| 1057/1057 [03:14<00:00,  5.42it/s]


Zero-shot Fold 2: {'accuracy': 0.7369914853358562, 'precision_false': 0.6554621848739496, 'recall_false': 0.4431818181818182, 'f1_false': 0.5288135593220339}


Fold 3 | 0-shot: 100%|██████████| 1048/1048 [03:12<00:00,  5.44it/s]


Zero-shot Fold 3: {'accuracy': 0.7414122137404581, 'precision_false': 0.6419753086419753, 'recall_false': 0.4588235294117647, 'f1_false': 0.5351629502572899}


Fold 4 | 0-shot: 100%|██████████| 1042/1042 [03:11<00:00,  5.45it/s]

Zero-shot Fold 4: {'accuracy': 0.744721689059501, 'precision_false': 0.6551724137931034, 'recall_false': 0.44970414201183434, 'f1_false': 0.5333333333333333}
Zero-shot Average Metrics: {'accuracy': np.float64(0.7437144833408882), 'precision_false': np.float64(0.6584480683402513), 'recall_false': np.float64(0.4511946339462332), 'f1_false': np.float64(0.5354005898958719)}
----------------------------------------------------------------------------------------------------


In [49]:
# Run for one-shot
one_shot_metrics = []
for fold in range(5):
    metrics = classify_fold_with_prompts(fold, shot_number=1)
    one_shot_metrics.append(metrics)
    print(f"Few-shot Fold {fold}: {metrics}")

one_shot_avg = {k: np.mean([m[k] for m in few_shot_metrics]) for k in one_shot_metrics[0]}
print("One-shot Average Metrics:", one_shot_avg)
print("-" * 100)

Fold 0 | 1-shot: 100%|██████████| 1073/1073 [03:20<00:00,  5.36it/s]


Few-shot Fold 0: {'accuracy': 0.48928238583411, 'precision_false': 0.3595505617977528, 'recall_false': 0.735632183908046, 'f1_false': 0.4830188679245283}


Fold 1 | 1-shot: 100%|██████████| 1064/1064 [03:17<00:00,  5.38it/s]


Few-shot Fold 1: {'accuracy': 0.45206766917293234, 'precision_false': 0.3654292343387471, 'recall_false': 0.8974358974358975, 'f1_false': 0.5193734542456719}


Fold 2 | 1-shot: 100%|██████████| 1057/1057 [03:18<00:00,  5.31it/s]


Few-shot Fold 2: {'accuracy': 0.336802270577105, 'precision_false': 0.3342830009496676, 'recall_false': 1.0, 'f1_false': 0.501067615658363}


Fold 3 | 1-shot: 100%|██████████| 1048/1048 [03:14<00:00,  5.39it/s]


Few-shot Fold 3: {'accuracy': 0.4074427480916031, 'precision_false': 0.3503727369542066, 'recall_false': 0.9676470588235294, 'f1_false': 0.5144644253322909}


Fold 4 | 1-shot: 100%|██████████| 1042/1042 [03:14<00:00,  5.36it/s]

Few-shot Fold 4: {'accuracy': 0.5786948176583493, 'precision_false': 0.4207221350078493, 'recall_false': 0.7928994082840237, 'f1_false': 0.5497435897435897}
Few-shot Average Metrics: {'accuracy': np.float64(0.45285797826681995), 'precision_false': np.float64(0.36607153380964463), 'recall_false': np.float64(0.8787229096902994), 'f1_false': np.float64(0.5135335905808888)}
----------------------------------------------------------------------------------------------------


In [50]:
# Run for few-shot
few_shot_metrics = []
for fold in range(5):
    metrics = classify_fold_with_prompts(fold, shot_number=2)
    few_shot_metrics.append(metrics)
    print(f"Few-shot Fold {fold}: {metrics}")

few_shot_avg = {k: np.mean([m[k] for m in few_shot_metrics]) for k in few_shot_metrics[0]}
print("Few-shot Average Metrics:", few_shot_avg)
print("-" * 100)

Fold 0 | 2-shot: 100%|██████████| 1073/1073 [03:26<00:00,  5.20it/s]


Few-shot Fold 0: {'accuracy': 0.40354147250698974, 'precision_false': 0.3399122807017544, 'recall_false': 0.8908045977011494, 'f1_false': 0.49206349206349204}


Fold 1 | 2-shot: 100%|██████████| 1064/1064 [03:25<00:00,  5.18it/s]


Few-shot Fold 1: {'accuracy': 0.3618421052631579, 'precision_false': 0.3401559454191033, 'recall_false': 0.9943019943019943, 'f1_false': 0.5068990559186638}


Fold 2 | 2-shot: 100%|██████████| 1057/1057 [03:25<00:00,  5.14it/s]


Few-shot Fold 2: {'accuracy': 0.336802270577105, 'precision_false': 0.3339676498572788, 'recall_false': 0.9971590909090909, 'f1_false': 0.5003563791874555}


Fold 3 | 2-shot: 100%|██████████| 1048/1048 [03:24<00:00,  5.13it/s]


Few-shot Fold 3: {'accuracy': 0.3416030534351145, 'precision_false': 0.32741617357001973, 'recall_false': 0.9764705882352941, 'f1_false': 0.49039881831610044}


Fold 4 | 2-shot: 100%|██████████| 1042/1042 [03:19<00:00,  5.21it/s]

Few-shot Fold 4: {'accuracy': 0.3963531669865643, 'precision_false': 0.3440514469453376, 'recall_false': 0.9497041420118343, 'f1_false': 0.5051140833988985}
Few-shot Average Metrics: {'accuracy': np.float64(0.36802841375378625), 'precision_false': np.float64(0.33710069929869874), 'recall_false': np.float64(0.9616880826318726), 'f1_false': np.float64(0.49896636577692205)}
----------------------------------------------------------------------------------------------------


**DISCUSS Q2 PART A HERE.**

In [ ]:
# IMPLEMENT Q2 PART B HERE.
# YOU CAN ADD CELLS BELOW.

**DISCUSS Q2 PART B HERE.**

In [ ]:
# IMPLEMENT Q2 PART C HERE.
# YOU CAN ADD CELLS BELOW.

**DISCUSS Q2 PART C HERE.**

## Q3 - Discussion of classification performance and resource use (20 points)

In this part, you are expected to discuss the misinformation detection performance and resource use of the models you have trained in Q1 Part A-B and Q2 Part A-B. Which model performs better? Which model uses more resources? Please discuss your OWN findings with YOUR OWN WORDS. Do not forget to share the time and resource need details you have measured and observed.

**WRITE YOUR DISCUSSION FOR Q3 HERE.**